用pytorch写RNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
device="cuda" if torch.cuda.is_available() else "cpu"



#手动实现RNNCELL

In [ ]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size,output_size):
        super.__init__()
        self.hidden_size = hidden_size
        # RNN 核心参数
        self.Wxh = nn.Parameter(torch.empty(input_size, hidden_size))    # x -> h
        self.Whh = nn.Parameter(torch.empty(hidden_size, hidden_size))   # h -> h
        self.bh  = nn.Parameter(torch.zeros(hidden_size))                # hidden bias

        # 输出层参数
        self.Why = nn.Parameter(torch.empty(hidden_size, output_size))   # h -> y
        self.by  = nn.Parameter(torch.zeros(output_size))                # output bias



In [ ]:
def forward(self,x,h_prev):#h_prev前一时刻隐藏状态
    h=torch.tanh(torch.mm(x,self.Wxh)+torch.mm(h_prev,self.Whh)+self.bh)
    y=torch.mm(h,self.Why)+self.by
    return y,h


In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class ManualRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size

        # RNN 参数
        self.Wxh = nn.Parameter(torch.empty(input_size, hidden_size))
        self.Whh = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.bh  = nn.Parameter(torch.zeros(hidden_size))

        # 输出层参数
        self.Why = nn.Parameter(torch.empty(hidden_size, output_size))
        self.by  = nn.Parameter(torch.zeros(output_size))

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.Wxh)
        nn.init.orthogonal_(self.Whh)
        nn.init.xavier_uniform_(self.Why)

    def forward(self, x):
        # x: [batch, seq_len, input_size]
        batch_size, seq_len, _ = x.shape
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)

        for t in range(seq_len):
            x_t = x[:, t, :]  # [batch, input_size]
            h = torch.tanh(x_t @ self.Wxh + h @ self.Whh + self.bh)

        logits = h @ self.Why + self.by  # [batch, output_size]
        return logits


def make_batch(batch_size, seq_len, device):
    """
    任务：输入0/1序列，预测1的个数是奇数还是偶数
    标签：0=偶数, 1=奇数
    """
    x = torch.randint(0, 2, (batch_size, seq_len, 1), device=device).float()
    y = (x.sum(dim=1).squeeze(-1) % 2).long()
    return x, y


def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 超参数
    input_size = 1
    hidden_size = 32
    output_size = 2
    seq_len = 10
    batch_size = 64
    epochs = 300
    lr = 0.1

    model = ManualRNN(input_size, hidden_size, output_size).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)

    # 训练
    for epoch in range(1, epochs + 1):
        x_batch, y_batch = make_batch(batch_size, seq_len, device)

        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if epoch % 50 == 0:
            with torch.no_grad():
                pred = logits.argmax(dim=1)
                acc = (pred == y_batch).float().mean().item()
            print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | Acc: {acc:.4f}")

    # 测试
    model.eval()
    with torch.no_grad():
        x_test, y_test = make_batch(1000, seq_len, device)
        pred = model(x_test).argmax(dim=1)
        test_acc = (pred == y_test).float().mean().item()
    print(f"Test Acc: {test_acc:.4f}")


if __name__ == "__main__":
    main()

Epoch  50 | Loss: 0.6976 | Acc: 0.4844
Epoch 100 | Loss: 0.6876 | Acc: 0.5156
Epoch 150 | Loss: 0.7039 | Acc: 0.4844
Epoch 200 | Loss: 0.7009 | Acc: 0.4844
Epoch 250 | Loss: 0.6999 | Acc: 0.4688
Epoch 300 | Loss: 0.7069 | Acc: 0.5312
Test Acc: 0.4980
